# C-MAPSS RUL Prediction — Visualization

This notebook visualizes the ML outputs produced by `02_analytics_ml.ipynb`.

**Input tables**
- `workspace.default.ml_model_results`
- `workspace.default.ml_feature_importance`
- `workspace.default.ml_validation_predictions`
- `workspace.default.ml_test_predictions`


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt

model_results = spark.table("workspace.default.ml_model_results")
feature_importance = spark.table("workspace.default.ml_feature_importance")
validation_predictions = spark.table("workspace.default.ml_validation_predictions")
test_predictions = spark.table("workspace.default.ml_test_predictions")

print("All four ML output tables loaded successfully.")
print("Model results:", model_results.count())
print("Feature importance:", feature_importance.count())
print("Validation predictions:", validation_predictions.count())
print("Test predictions:", test_predictions.count())


## 1. Model Performance Comparison

In [0]:
display(model_results.orderBy("RMSE"))


In [0]:
pdf = model_results.orderBy("RMSE").toPandas()

ax = pdf.plot(
    x="model", y="RMSE", kind="bar",
    legend=False, figsize=(8,5), rot=0
)
ax.set_title("Model Comparison — RMSE (Lower is Better)")
ax.set_xlabel("Model")
ax.set_ylabel("RMSE")
plt.tight_layout()
plt.show()


In [0]:
pdf = model_results.orderBy(F.col("R2").desc()).toPandas()

ax = pdf.plot(
    x="model", y="R2", kind="bar",
    legend=False, figsize=(8,5), rot=0
)
ax.set_title("Model Comparison — R² (Higher is Better)")
ax.set_xlabel("Model")
ax.set_ylabel("R²")
plt.tight_layout()
plt.show()


## 2. Feature / Sensor Importance

In [0]:
display(
    feature_importance
    .orderBy(F.col("importance").desc())
    .limit(15)
)


In [0]:
fi = (
    feature_importance
    .orderBy(F.col("importance").desc())
    .limit(15)
    .toPandas()
    .sort_values("importance")
)

ax = fi.plot(
    x="feature", y="importance",
    kind="barh", legend=False, figsize=(9,6)
)
ax.set_title("Top 15 Features / Sensors by Importance")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()


## 3. Actual vs Predicted RUL

In [0]:
display(validation_predictions.limit(5000))


In [0]:
vp = (
    validation_predictions
    .select("actual_RUL", "prediction")
    .dropna()
    .limit(5000)
    .toPandas()
)

ax = vp.plot(
    x="actual_RUL", y="prediction",
    kind="scatter", figsize=(7,7), alpha=0.35
)

lo = min(vp["actual_RUL"].min(), vp["prediction"].min())
hi = max(vp["actual_RUL"].max(), vp["prediction"].max())

ax.plot([lo, hi], [lo, hi], linestyle="--")
ax.set_title("Actual vs Predicted RUL — Validation Set")
ax.set_xlabel("Actual RUL")
ax.set_ylabel("Predicted RUL")
plt.tight_layout()
plt.show()


## 4. Prediction Error Distribution

In [0]:
error_df = (
    validation_predictions
    .select(
        (F.col("prediction") - F.col("actual_RUL")).alias("error")
    )
    .dropna()
    .limit(10000)
    .toPandas()
)

ax = error_df["error"].plot(
    kind="hist", bins=40, figsize=(8,5)
)
ax.set_title("Prediction Error Distribution")
ax.set_xlabel("Prediction Error (Predicted − Actual RUL)")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()


## 5. Final Test RUL by Engine

In [0]:
# One final prediction per engine: prediction at its final observed test cycle.
final_test = (
    test_predictions
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("unit_number")
            .orderBy(F.col("cycle").desc())
        )
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
    .orderBy("unit_number")
)

display(final_test)


In [0]:
ft = final_test.select("unit_number", "prediction").toPandas()

ax = ft.plot(
    x="unit_number", y="prediction",
    kind="bar", legend=False, figsize=(12,5)
)
ax.set_title("Predicted Remaining Useful Life by Engine")
ax.set_xlabel("Engine / Unit Number")
ax.set_ylabel("Predicted RUL")
plt.tight_layout()
plt.show()


## 6. Dashboard KPI Tables

Use these outputs when building the final Databricks dashboard.


In [0]:
# Best-performing model based on lowest RMSE
display(model_results.orderBy("RMSE").limit(1))


In [0]:
# Top 10 most important features/sensors
display(
    feature_importance
    .orderBy(F.col("importance").desc())
    .limit(10)
)


In [0]:
# Engines ordered from lowest to highest predicted RUL
display(
    final_test
    .select("unit_number", "prediction")
    .orderBy(F.col("prediction"))
)


## Recommended Final Dashboard

1. Best Model / RMSE / R² KPI cards
2. RMSE comparison
3. R² comparison
4. Top sensor/feature importance
5. Actual vs Predicted RUL scatter plot
6. Prediction error distribution
7. Predicted RUL by engine

**Do not retrain models in this notebook.** This notebook only visualizes the ML output tables.
